### Notebook Summary: Property Recommender System

In this notebook, we built a local property recommendation engine using LangChain and ChromaDB. Here's a summary of the steps performed:

1.  **Environment Setup**: Installed necessary libraries including `langchain`, `langchain-huggingface`, `langchain-chroma`, and `sentence-transformers`.
2.  **Data Loading**: Loaded and inspected a cleaned dataset of property listings (`nawy_properties_cleaned.csv`) using `pandas` and LangChain's `CSVLoader`.
3.  **Metadata Configuration**: Configured the loader to preserve key property attributes (id, location, price, beds, etc.) as metadata for better filtering and display.
4.  **Document Chunking**: Utilized a `CharacterTextSplitter` to prepare the property descriptions for embedding while maintaining individual listing integrity.
5.  **Local Embeddings**: Initialized a high-performance local embedding model (`Qwen3-Embedding-0.6B`) to convert text into vector representations.
6.  **Vector Database**: Created a `Chroma` vectorstore to store the embedded documents and persisted the database locally to the `./property_recommender_db` directory.
7.  **Semantic Search & Retrieval**: Implemented a retrieval system to find the top 5 most relevant properties based on natural language queries (e.g., searching for chalets in Sidi Heneish or duplexes in October City).
8.  **Export**: Compressed the persistent database into a ZIP file and provided a download link for portability.

In [1]:
%pip install langchain langchain-community langchain-huggingface langchain-chroma sentence-transformers pandas

In [2]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import pandas as pd

In [3]:
df = pd.read_csv("nawy_properties_cleaned.csv")
df.columns

Index(['id', 'location', 'property_name', 'description', 'm2', 'Beds', 'Baths',
       'payment_plan', 'price', 'url_path', 'tag', 'cover_image',
       'developer_logo', 'price_float', 'embedding_text'],
      dtype='object')

In [4]:
max_len = df['embedding_text'].str.len().max()
print(f"The maximum length in 'embedding_text' is: {max_len}")

The maximum length in 'embedding_text' is: 304


In [5]:
# Re-load CSV with 'location' included in metadata
loader = CSVLoader(
    file_path="nawy_properties_cleaned.csv",
    source_column="embedding_text",
    metadata_columns=["id", "location", "property_name", "m2", "Beds", "Baths", "price_float"]
)
docs = loader.load()

In [6]:
# Chunk with no overlap - splits each property row on newlines only if multi-line
text_splitter = CharacterTextSplitter(
    separator="\n\n",  # Split on double newlines (paragraphs) or "\n" for lines
    chunk_size=500,
    chunk_overlap=0  # No overlap as requested
)
splits = text_splitter.split_documents(docs)

In [7]:
# Embeddings - local Qwen model (downloads on first use ~1.19G)
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"trust_remote_code": True}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [8]:
# Persist to Chroma - best for local dev/production similarity search
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./property_recommender_db"  # Saves locally
)

In [9]:
# Retrieve recommendations
query = "Chalet in Sidi Heneish near the beach"
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})  # Top 5 similar
recommendations = retriever.invoke(query)

for doc in recommendations:
    print(f"Property: {doc.metadata.get('id')}")
    print(f"Text: {doc.page_content[:]}...")
    print("---")

Property: 03738
Text: description: Chalet for sale in Summer  - with 2 bedrooms in Sidi Heneish by Al Ahly Sabbour Developments.
payment_plan: 
price: 12,796,403 EGP
url_path: https://www.nawy.com/compound/1161-summer/property/63577-chalet-for-sale-in-summer
tag: Resale
cover_image: https://prod-images.cooingestate.com/processed/property_image/image/320373/high.webp
developer_logo: https://prod-images.cooingestate.com/processed/developer/logo_image/191/medium.webp
embedding_text: id: 03738 | location: Sidi Heneish | property_name: Chalet, Summer | description: Chalet for sale in Summer  - with 2 bedrooms in Sidi Heneish by Al Ahly Sabbour Developments. | Beds: 2.0 | Baths: 2.0 | m2: 95.0 | tag: Resale...
---
Property: 03769
Text: description: Chalet for sale in Summer  - with 2 bedrooms in Sidi Heneish by Al Ahly Sabbour Developments.
payment_plan: 
price: 12,953,028 EGP
url_path: https://www.nawy.com/compound/1161-summer/property/70055-chalet-for-sale-in-summer
tag: Resale
cover_image

In [13]:
# Retrieve recommendations
query = "2 or 3 bedroom duplex in October City"
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})  # Top 5 similar
recommendations = retriever.invoke(query)

for doc in recommendations:
    print(f"Property: {doc.metadata.get('id')}")
    print(f"Location: {doc.metadata.get('location')}")
    print(f"Price: {doc.metadata.get('price_float')}")
    print(f"Property Name: {doc.metadata.get('property_name')}")
    print(f"Text: {doc.page_content[:]}...")
    print("---")

Property: 04854
Location: 6th of October City
Price: 14800000.0
Property Name: Duplex, Mountain View October Park
Text: description: Duplex for sale in Mountain View October Park - with 3 bedrooms in 6th of October City by Mountain View.
payment_plan: 
price: 14,800,000 EGP
url_path: https://www.nawy.com/compound/10-mountain-view-october-park/property/62078-duplex-for-sale-in-mountain-view-october-park-with-3-bedrooms-in-6th-of-october-city-by-mountain-view
tag: Resale
cover_image: https://prod-images.cooingestate.com/processed/inventory/unlockedProperties/62078/57a660e5-c81b-427e-9c9c-6bedb9063e69/high.webp
developer_logo: https://prod-images.cooingestate.com/processed/developer/logo_image/6/medium.webp
embedding_text: id: 04854 | location: 6th of October City | property_name: Duplex, Mountain View October Park | description: Duplex for sale in Mountain View October Park - with 3 bedrooms in 6th of October City by Mountain View. | Beds: 3.0 | Baths: 4.0 | m2: 210.0 | tag: Resale...
--

In [14]:
!zip -r property_recommender_db.zip property_recommender_db

  adding: property_recommender_db/ (stored 0%)
  adding: property_recommender_db/chroma.sqlite3 (deflated 67%)
  adding: property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/ (stored 0%)
  adding: property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/index_metadata.pickle (deflated 45%)
  adding: property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/length.bin (deflated 99%)
  adding: property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/data_level0.bin (deflated 54%)
  adding: property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/header.bin (deflated 55%)
  adding: property_recommender_db/ee6ae8fd-8016-47d1-a235-ebd25259d855/link_lists.bin (deflated 82%)


In [15]:
from google.colab import files
files.download("property_recommender_db.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>